In [1]:
!pip install -q transformers torch pandas


# Laboratorio 4

[Link Repositorio](https://github.com/donmatthiuz/NLP/tree/lab4)

## Imports

In [2]:
import pandas as pd
from transformers import pipeline
import json




## Dataset

El dataset Dataset for Sentiment Analysis in Spanish, creado por Francisco José Ramírez Vicente. Contiene reseñas obtenidas de Google Maps y Tripadvisor, con las columnas puntuación, review y sentimiento. Incluye opiniones positivas, negativas, neutrales/mixtas, negaciones y menciones de personas y lugares.

In [3]:

url = "https://raw.githubusercontent.com/fjramirezv/sentiment-webscraping/main/dataset_sentiment_analisys.csv"

df = pd.read_csv(
    url,
    sep=";",
    encoding="utf-8-sig",
    engine="python",
    on_bad_lines="skip"
)

# Cambiar nombres de columnas
df = df.rename(columns={
    "puntuación": "puntuacion",
    "review": "reseña"
})

# Eliminar filas vacías
df = df.dropna(subset=["reseña", "sentimiento"])
df = df[df["reseña"].str.strip() != ""]

# Mantener ejemplos de cada sentimiento
obligatorias = pd.concat([
    df[df["sentimiento"] == "positivo"].head(10),
    df[df["sentimiento"] == "negativo"].head(5),
    df[df["sentimiento"] == "neutral"].head(5)
])

restantes = df.drop(obligatorias.index)
cantidad_faltante = 30 - len(obligatorias)

dataset_30 = pd.concat([
    obligatorias,
    restantes.head(cantidad_faltante)
]).head(30).reset_index(drop=True)

# Considerar las opiniones neutrales con aspectos buenos y malos como mixtas
dataset_30["tipo"] = dataset_30["sentimiento"].replace({
    "neutral": "mixto"
})

# Detectar posibles negaciones
dataset_30["tiene_negacion"] = dataset_30["reseña"].str.contains(
    r"\b(no|nunca|jamás|tampoco|ni|sin)\b",
    case=False,
    regex=True
)

display(dataset_30)

/tmp/ipykernel_2801/562147463.py:42: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  dataset_30["tiene_negacion"] = dataset_30["reseña"].str.contains(


,puntuacion,reseña,sentimiento,tipo,tiene_negacion
0,40.0,Dispone de espacios acogedores con unas mampar...,positivo,positivo,False
1,40.0,"Tanto la comida como los pintxos son buenos, l...",positivo,positivo,False
2,50.0,Hemos comido estupendamente en la terraza con ...,positivo,positivo,False
3,50.0,Excelente comida en una de las mejores terraza...,positivo,positivo,False
4,40.0,"Para comer o cenar algo esta bien, lo malo es ...",positivo,positivo,False
5,40.0,un sitio muy agradable para tomar un café unas...,positivo,positivo,False
6,50.0,Conocí este sitio a través de internet y me pa...,positivo,positivo,False
7,50.0,Tras pedirme un café y le pedí consejo al cama...,positivo,positivo,False
8,50.0,Sitio agradable y familiar . El trato por el p...,positivo,positivo,False
9,50.0,El bully nunca falla como escenario para nuest...,positivo,positivo,True


## 5. Parte A: Sentimiento

In [4]:

# Modelo de sentimiento entrenado para textos en español
sentiment = pipeline(
    "sentiment-analysis",
    model="pysentimiento/robertuito-sentiment-analysis"
)

# Convertir la columna de reseñas en una lista
reseñas = dataset_30["reseña"].fillna("").astype(str).tolist()

# Procesar las reseñas en lotes de 8
resultados = sentiment(
    reseñas,
    batch_size=8,
    truncation=True
)

# Traducir las etiquetas del modelo
etiquetas = {
    "POS": "positivo",
    "NEG": "negativo",
    "NEU": "neutral"
}

# Guardar los resultados por reseña
df_resultados = pd.DataFrame([
    {
        "identificador": indice + 1,
        "texto_original": texto,
        "etiqueta_sentimiento": etiquetas.get(
            resultado["label"],
            resultado["label"]
        ),
        "score_confianza": round(float(resultado["score"]), 4)
    }
    for indice, (texto, resultado) in enumerate(zip(reseñas, resultados))
])

display(df_resultados)


# Guaradarlos en un csv
df_resultados.to_csv(
    "resultados_sentimiento.csv",
    index=False,
    encoding="utf-8-sig"
)

config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  435MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

,identificador,texto_original,etiqueta_sentimiento,score_confianza
0,1,Dispone de espacios acogedores con unas mampar...,positivo,0.9288
1,2,"Tanto la comida como los pintxos son buenos, l...",positivo,0.9817
2,3,Hemos comido estupendamente en la terraza con ...,positivo,0.9789
3,4,Excelente comida en una de las mejores terraza...,positivo,0.9791
4,5,"Para comer o cenar algo esta bien, lo malo es ...",negativo,0.7070
5,6,un sitio muy agradable para tomar un café unas...,positivo,0.9508
6,7,Conocí este sitio a través de internet y me pa...,positivo,0.9429
7,8,Tras pedirme un café y le pedí consejo al cama...,positivo,0.9300
8,9,Sitio agradable y familiar . El trato por el p...,positivo,0.9735
9,10,El bully nunca falla como escenario para nuest...,positivo,0.9764


## 6. Parte B: NER


In [5]:

## Pipeline
ner = pipeline(
    "token-classification",
    model="Davlan/bert-base-multilingual-cased-ner-hrl",
    aggregation_strategy="simple"
)

# Utiliza la misma lista de reseñas de la Parte A
reseñas = dataset_30["reseña"].fillna("").astype(str).tolist()

# Procesar las reseñas en lote
resultados_ner = ner(
    reseñas,
    batch_size=8,
)

config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  709MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/264 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [6]:
entidades_extraidas = []

for identificador, (texto, entidades) in enumerate(
    zip(reseñas, resultados_ner),
    start=1
):
    for entidad in entidades:

        # La clave depende del modelo
        tipo = entidad.get(
            "entity_group",
            entidad.get("entity", "DESCONOCIDO")
        )

        entidades_extraidas.append({
            "identificador_reseña": identificador,
            "texto_original": texto,
            "texto_entidad": entidad.get("word", "").strip(),
            "tipo_entidad": tipo,
            "score": round(float(entidad.get("score", 0)), 4)
        })

df_entidades = pd.DataFrame(
    entidades_extraidas,
    columns=[
        "identificador_reseña",
        "texto_original",
        "texto_entidad",
        "tipo_entidad",
        "score"
    ]
)

display(df_entidades)

,identificador_reseña,texto_original,texto_entidad,tipo_entidad,score
0,4,Excelente comida en una de las mejores terraza...,Donosti,LOC,0.9993
1,20,"Es un sitio genial, con camareros muy atentos ...",Claudia,PER,0.9909
2,22,"Recomendable si quieres sentirte como en casa,...",Ibai,PER,0.9945
3,22,"Recomendable si quieres sentirte como en casa,...",Lander,PER,0.9992
4,22,"Recomendable si quieres sentirte como en casa,...",Iker,PER,0.9993
5,23,"Comer en Donosti, al sol en la terraza y en No...",Donosti,LOC,0.9987
6,23,"Comer en Donosti, al sol en la terraza y en No...",Nochebuena,LOC,0.9474
7,23,"Comer en Donosti, al sol en la terraza y en No...",Lander,PER,0.9987
8,23,"Comer en Donosti, al sol en la terraza y en No...",Ibai,PER,0.9979
9,24,Servicio rápido y de calidad. Perfecta atenció...,Ibai,PER,0.9427


In [7]:

## GUardarlo
df_entidades.to_csv(
    "resultados_ner.csv",
    index=False,
    encoding="utf-8-sig"
)

## 7. Parte C: Flujo Integrado


In [8]:

mapa_sentimiento = {
    "POS": "POSITIVE",
    "NEG": "NEGATIVE",
    "NEU": "NEUTRAL"
}

resultados_combinados = []

for review_id, (texto, sentimiento, entidades) in enumerate(
    zip(reseñas, resultados, resultados_ner),
    start=1
):
    etiqueta_original = sentimiento.get("label", "UNKNOWN")

    entidades_formateadas = []

    for entidad in entidades:
        # La etiqueta puede variar según el modelo
        etiqueta_entidad = entidad.get(
            "entity_group",
            entidad.get("entity", "UNKNOWN")
        )

        entidades_formateadas.append({
            "label": etiqueta_entidad,
            "text": entidad.get("word", "").strip(),
            "score": round(float(entidad.get("score", 0)), 4)
        })

    resultados_combinados.append({
        "review_id": review_id,
        "text": texto,
        "sentiment": mapa_sentimiento.get(
            etiqueta_original,
            etiqueta_original
        ),
        "sentiment_score": round(
            float(sentimiento.get("score", 0)),
            4
        ),
        "entities": entidades_formateadas
    })

In [9]:
## Json del primer registro

primera_con_entidades = next(
    (
        resultado
        for resultado in resultados_combinados
        if len(resultado["entities"]) > 0
    ),
    None
)

if primera_con_entidades:
    print(
        json.dumps(
            primera_con_entidades,
            ensure_ascii=False,
            indent=4
        )
    )
else:
    print("No se encontraron reseñas con entidades.")

{
    "review_id": 4,
    "text": "Excelente comida en una de las mejores terrazas de Donosti. Atención de 10. Una maravillosa opción en un lugar tranquiloMás",
    "sentiment": "POSITIVE",
    "sentiment_score": 0.9791,
    "entities": [
        {
            "label": "LOC",
            "text": "Donosti",
            "score": 0.9993
        }
    ]
}


## 8. Parte D: Estadísticas


In [10]:

# Resumen de una fila por reseña, construido desde el flujo integrado
df_resumen = pd.DataFrame([
    {
        "review_id": resultado["review_id"],
        "text": resultado["text"],
        "sentiment": resultado["sentiment"],
        "sentiment_score": resultado["sentiment_score"],
        "num_entidades": len(resultado["entities"]),
        "tipos_entidad": ", ".join(
            sorted({entidad["label"] for entidad in resultado["entities"]})
        ),
        "entidades": json.dumps(resultado["entities"], ensure_ascii=False)
    }
    for resultado in resultados_combinados
])

display(df_resumen)


,review_id,text,sentiment,sentiment_score,num_entidades,tipos_entidad,entidades
0,1,Dispone de espacios acogedores con unas mampar...,POSITIVE,0.9288,0,,[]
1,2,"Tanto la comida como los pintxos son buenos, l...",POSITIVE,0.9817,0,,[]
2,3,Hemos comido estupendamente en la terraza con ...,POSITIVE,0.9789,0,,[]
3,4,Excelente comida en una de las mejores terraza...,POSITIVE,0.9791,1,LOC,"[{""label"": ""LOC"", ""text"": ""Donosti"", ""score"": ..."
4,5,"Para comer o cenar algo esta bien, lo malo es ...",NEGATIVE,0.7070,0,,[]
5,6,un sitio muy agradable para tomar un café unas...,POSITIVE,0.9508,0,,[]
6,7,Conocí este sitio a través de internet y me pa...,POSITIVE,0.9429,0,,[]
7,8,Tras pedirme un café y le pedí consejo al cama...,POSITIVE,0.9300,0,,[]
8,9,Sitio agradable y familiar . El trato por el p...,POSITIVE,0.9735,0,,[]
9,10,El bully nunca falla como escenario para nuest...,POSITIVE,0.9764,0,,[]


In [11]:

# Cantidad de reseñas por etiqueta
print("Reseñas por etiqueta:")
print(df_resumen["sentiment"].value_counts().to_string())

# Score promedio global y por etiqueta
print("\nScore promedio global:", round(df_resumen["sentiment_score"].mean(), 4))

print("\nScore promedio por etiqueta:")
print(
    df_resumen.groupby("sentiment")["sentiment_score"]
    .mean()
    .round(4)
    .to_string()
)


Reseñas por etiqueta:
sentiment
POSITIVE    21
NEGATIVE     9

Score promedio global: 0.9575

Score promedio por etiqueta:
sentiment
NEGATIVE    0.9280
POSITIVE    0.9701


In [12]:

# Cinco reseñas de menor confianza
menor_confianza = df_resumen.nsmallest(5, "sentiment_score")

display(
    menor_confianza[["review_id", "sentiment", "sentiment_score", "text"]]
)


,review_id,sentiment,sentiment_score,text
4,5,NEGATIVE,0.7070,"Para comer o cenar algo esta bien, lo malo es ..."
12,13,NEGATIVE,0.8954,"Un poco decepción, después de las buenas recom..."
13,14,NEGATIVE,0.8954,"Un poco decepción, después de las buenas recom..."
0,1,POSITIVE,0.9288,Dispone de espacios acogedores con unas mampar...
7,8,POSITIVE,0.9300,Tras pedirme un café y le pedí consejo al cama...


In [13]:

# Número promedio de entidades por reseña
print("Entidades totales:", int(df_resumen["num_entidades"].sum()))
print(
    "Promedio de entidades por reseña:",
    round(df_resumen["num_entidades"].mean(), 4)
)
print(
    "Reseñas sin ninguna entidad:",
    int((df_resumen["num_entidades"] == 0).sum())
)

# Tipos de entidad encontrados
print("\nTipos de entidad encontrados:")
print(df_entidades["tipo_entidad"].value_counts().to_string())

print("\nScore promedio por tipo de entidad:")
print(
    df_entidades.groupby("tipo_entidad")["score"]
    .mean()
    .round(4)
    .to_string()
)


Entidades totales: 12
Promedio de entidades por reseña: 0.4
Reseñas sin ninguna entidad: 23

Tipos de entidad encontrados:
tipo_entidad
PER    9
LOC    3

Score promedio por tipo de entidad:
tipo_entidad
LOC    0.9818
PER    0.9909


In [14]:

# Exportar el resumen
df_resumen.to_csv(
    "laboratorio_4_resultados.csv",
    index=False,
    encoding="utf-8-sig"
)


## 9. Parte E: Análisis de casos difíciles


In [15]:

# Reseñas elegidas para la revisión manual
casos_dificiles = [5, 10, 15, 17, 23, 25, 29]

for review_id in casos_dificiles:
    resultado = resultados_combinados[review_id - 1]

    print("Reseña", resultado["review_id"])
    print(resultado["text"])
    print(
        "Sentimiento:",
        resultado["sentiment"],
        "| score:",
        resultado["sentiment_score"]
    )
    print(
        "Entidades:",
        resultado["entities"] if resultado["entities"] else "ninguna"
    )
    print("-" * 80)


Reseña 5
Para comer o cenar algo esta bien, lo malo es que las mesas estas un poco cerca una de la otra y con carro es un poco dificil acceder, en cuanto a comida hay entrantes, hamburguesas, pasta, y platos combinados. La torrija caramelizada un 10.Más
Sentimiento: NEGATIVE | score: 0.707
Entidades: ninguna
--------------------------------------------------------------------------------
Reseña 10
El bully nunca falla como escenario para nuestras cenas de amigas del equipo de baloncesto, y ya son desde hace 11 años que nos reunimos aqui, como siempre el servicio espectacular y la comida inmejorable. Muy recomendable!Más
Sentimiento: POSITIVE | score: 0.9764
Entidades: ninguna
--------------------------------------------------------------------------------
Reseña 15
El lugar muy bonito, la atención buena. Pedimos un calamar a la plancha para picar y era UN calamar con poco sabor. Para plato principal pedimos fideuá… tristísima. Realmente prefiero hacer unos fideos con salsa de tomate en

### Caso 1. Reseña 5, opinión mixta con el score más bajo

Texto: "Para comer o cenar algo esta bien, lo malo es que las mesas estas un poco cerca una de la otra y con carro es un poco dificil acceder... La torrija caramelizada un 10."

- Predicción del modelo: negativo, score 0.7070. Es el score más bajo de las 30 reseñas.
- Interpretación humana: opinión mixta con saldo positivo. El dataset original la etiqueta como positivo con puntuación 4 de 5.
- ¿Coinciden? No.
- Posible causa: el modelo le da más peso al bloque de quejas ("lo malo es que", "un poco dificil") que al cierre positivo. Como en la práctica sólo usa dos clases, tiene que elegir un extremo, y el score bajo es justamente la señal de esa duda.

### Caso 2. Reseña 10, negación

Texto: "El bully nunca falla como escenario para nuestras cenas de amigas del equipo de baloncesto..."

- Predicción del modelo: positivo, score 0.9764. Sin entidades detectadas.
- Interpretación humana: positivo. "nunca falla" es una negación que invierte el significado a "siempre funciona".
- ¿Coinciden? Sí en sentimiento, no en NER.
- Posible causa: resuelve bien la negación porque el resto del texto es explícitamente positivo ("servicio espectacular", "comida inmejorable"), así que no depende de entenderla. El fallo real está en NER: "El bully" es el nombre del local y debería ser ORG, pero se escribe en minúscula y el modelo multilingüe no lo reconoce como nombre propio.

### Caso 3. Reseña 15, opinión mixta con confianza alta

Texto: "El lugar muy bonito, la atención buena. Pedimos un calamar a la plancha para picar y era UN calamar con poco sabor. Para plato principal pedimos fideuá... tristísima."

- Predicción del modelo: negativo, score 0.9723.
- Interpretación humana: opinión mixta. El lugar y la atención son buenos, la comida es mala. El dataset la marca como neutral con puntuación 2 de 5, y en este notebook se reclasificó como mixto.
- ¿Coinciden? Parcialmente. El saldo final sí es negativo, pero la etiqueta borra por completo la parte positiva.
- Posible causa: el modelo colapsa un texto de dos polaridades en una sola clase, y además lo hace con score muy alto. Es un error más peligroso que el del caso 1 porque el score no avisa de nada.

### Caso 4. Reseña 17, ironía

Texto: 'Desde luego una decepción. "Cria fama..." Mucha propaganda pero nada aconsejable. La paella era insípida, poco gustosa... El pan que sirven está bueno.'

- Predicción del modelo: negativo, score 0.9741.
- Interpretación humana: negativo, con una crítica irónica apoyada en el refrán "cría fama y échate a dormir".
- ¿Coinciden? Sí.
- Posible causa: acierta, pero no por interpretar la ironía, sino porque el texto trae marcadores léxicos explícitos ("decepción", "insípida", "nada aconsejable"). Un sarcasmo construido sólo con palabras positivas probablemente lo haría fallar.

### Caso 5. Reseña 23, entidad mal tipificada

Texto: "Comer en Donosti, al sol en la terraza y en Nochebuena, no tiene precio!! Muchas gracias a Lander y a Ibai por el genial servicio!"

- Predicción del modelo: positivo con score 0.9811. Entidades: Donosti LOC 0.9987, Nochebuena LOC 0.9474, Lander PER 0.9987, Ibai PER 0.9979.
- Interpretación humana: positivo. Donosti, Lander e Ibai están bien, pero Nochebuena es una fecha, no un lugar. Correspondería a MISC.
- ¿Coinciden? Sí en sentimiento, no en NER.
- Posible causa: "Nochebuena" va en mayúscula y aparece en la misma construcción de lugar que "en Donosti" unas palabras antes, así que el modelo copia el patrón sintáctico. Que le asigne 0.9474 a un error confirma que el score de NER tampoco está calibrado. De paso, "no tiene precio" es otra negación con sentido positivo que el modelo de sentimiento resolvió bien.

### Caso 6. Reseña 25, nombre en minúscula

Texto: "Servicio muy atento, todo de calidad y un buen precio, nos ha servido ibai, muy majo! Me encantan sus patatas bravas!"

- Predicción del modelo: positivo con score 0.9798. Ninguna entidad detectada.
- Interpretación humana: positivo, y "ibai" es claramente una persona, el camarero.
- ¿Coinciden? Sí en sentimiento, no en NER.
- Posible causa: el modelo de NER es "cased" y depende de la mayúscula inicial. El mismo nombre escrito "Ibai" sí se detecta en las reseñas 22, 23 y 24. Lo mismo pasa en la reseña 26 con "iker". En reseñas reales, escritas rápido y sin corregir, esto produce falsos negativos de forma sistemática.

### Caso 7. Reseña 29, opinión tibia y texto truncado

Texto: "Buen sitio para comer arroces en primera línea de playa... La comida está buena pero no me sorprendió especialmente. Esperaba las mejores paellas de..."

- Predicción del modelo: positivo, score 0.9640.
- Interpretación humana: opinión tibia, más cerca de neutral que de positiva. La puntuación original es 3 de 5 y el texto termina cortado justo donde empezaba la crítica.
- ¿Coinciden? No del todo.
- Posible causa: se suman dos problemas. El scraping truncó la reseña, así que el modelo nunca ve la parte negativa; y aun leyendo media opinión responde con 0.9640, sin reflejar en el score que le falta información.


## Aclaracion

Con la reseña 25 al volver a correr el notebook para terminar de hacer las preguntas de reflexion el valor de predicción cambio ligeramente y no coincide, pero es bastante similar, por lo que solo dejamos la puntuacion anterior con el mismo análisis.

## 10. Preguntas de reflexión

1. ¿El modelo parece apropiado para reseñas en español?
Sí, aunque con reservas. robertuito-sentiment-analysis está entrenado sobre texto en español y maneja bien el vocabulario coloquial, los acentos y los emojis de las reseñas, y acertó la polaridad dominante en casi todos los casos claros de las 30. El problema no es el idioma sino el dominio y el esquema de etiquetas: está entrenado con tuits, no con reseñas de restaurantes, y en estas 30 reseñas nunca usó la etiqueta neutral, así que toda opinión mixta terminó forzada a positivo o negativo. Para NER la respuesta es más floja, porque bert-base-multilingual-cased-ner-hrl no está especializado en español y se nota.

2. ¿Qué errores de sentimiento encontraste?
La reseña 5 es mixta con saldo positivo y salió negativo con 0.7070. Las reseñas 15 a 18, etiquetadas como neutral en el dataset, salieron todas negativo con score por encima de 0.97, perdiendo la parte positiva del texto. Las reseñas 29 y 30, con puntuación 3 de 5 y una crítica explícita, salieron positivo con 0.9640. El patrón de fondo es el mismo en los tres casos: no hay clase neutral, así que cualquier opinión matizada se convierte en un veredicto tajante. A eso se suma que varios textos vienen truncados por el scraping, terminados en "..." o en "Más", y el modelo clasifica sin ver el final de la opinión.

3. ¿Qué problemas observaste en NER?
Se extrajeron sólo 12 entidades en 30 reseñas, es decir 0.4 por reseña, y 23 reseñas quedaron sin ninguna. Sólo aparecieron dos tipos, PER y LOC, y ningún ORG, pese a que varias reseñas mencionan el local por su nombre. "Nochebuena" se clasificó como LOC con 0.9474, cuando es una fecha. Los nombres escritos en minúscula no se detectan: "ibai" en la reseña 25 e "iker" en la reseña 26 se pierden, mientras que "Ibai" e "Iker" con mayúscula sí se reconocen en otras reseñas. Y "el bully", que es el nombre del restaurante, nunca se reconoce como ORG en las reseñas 10 y 19. En resumen, el modelo depende demasiado de la mayúscula y funciona mal con nombres propios vascos y con nombres comerciales.

4. ¿Cómo influye el score en tu confianza?
Sirve, pero sólo en una dirección. Un score bajo sí fue buena señal de problema: el único valor de 0.7070 corresponde a la reseña 5, que efectivamente está mal clasificada. Al revés no funciona, porque las reseñas mixtas 15 a 18 están mal etiquetadas con scores de 0.97. Con un promedio global de 0.9575 y la mayoría de reseñas por encima de 0.95, el score casi no discrimina entre casos fáciles y difíciles. Hay que recordar que es un softmax sobre las clases del modelo, no una probabilidad calibrada de acierto. Lo mismo pasa en NER, donde el error de "Nochebuena" viene con 0.9474. Así que lo uso para ordenar y priorizar qué revisar a mano, no para decidir si una predicción es correcta.

5. ¿Qué datos etiquetarías manualmente para calcular precision, recall y F1?
Para sentimiento, un conjunto de reseñas del mismo dominio anotadas a mano con tres clases, positivo, negativo y neutral o mixto, razonablemente balanceado y con al menos dos anotadores para medir el acuerdo entre ellos. Sobremuestrearía justo los casos donde el modelo falla: opiniones mixtas, negaciones, ironía, textos muy cortos y textos truncados. También eliminaría los duplicados que trae el dataset, porque las reseñas 11 y 12, 13 y 14, 15 y 16, 17 y 18, y 29 y 30 están repetidas y inflarían las métricas. Para NER, las mismas reseñas anotadas entidad por entidad, marcando posición inicial, posición final y tipo, con PER, LOC, ORG y MISC, incluyendo a propósito nombres en minúscula, nombres de locales y nombres mal escritos, para poder calcular precision, recall y F1 por tipo de entidad.

6. ¿Usarías estas predicciones para tomar decisiones automáticas importantes? ¿Por qué?
No. Sí las usaría para tareas agregadas y de apoyo, como ordenar reseñas, detectar aumentos de quejas o enviar los casos dudosos a revisión humana. Pero no para decisiones con consecuencias reales, como penalizar a un local, evaluar a un empleado o publicar respuestas automáticas. Las razones son concretas: en la práctica el modelo sólo usa dos clases, así que toda opinión mixta se convierte en un veredicto tajante; el score no está calibrado y da 0.97 a predicciones equivocadas; el NER se pierde justamente los nombres de personas y locales, que es lo que haría falta para atribuir una crítica a alguien; y no tengo métricas medidas sobre datos etiquetados, sólo la inspección manual de 30 reseñas. Como mínimo haría falta un conjunto de prueba anotado, un umbral de confianza alto y revisión humana obligatoria antes de cualquier acción.
